# PennyLane: your first quantum circuit

A very basic, 10-minute tutorial. You only need a little Python. Run the cells from top to bottom in a Jupyter notebook or VS Code.

PennyLane is a Python library for quantum computing. Here we run everything on a **local simulator**, so you do not need a quantum computer or an account.

You will create a qubit, apply gates, read probabilities, and create an entangled pair.


## 1. Setup

If PennyLane is missing, uncomment and run the next cell. `%pip` installs into the notebook's Python environment. Restart the kernel after installation if needed.


In [ ]:
# %pip install pennylane


In [1]:
import pennylane as qml
import math

print("PennyLane version:", qml.__version__)


PennyLane version: 0.45.1


## 2. Your first qubit

A classical bit is 0 or 1. A qubit can be in a superposition of the states written `|0>` and `|1>`. When measured in this basis, it gives either 0 or 1.

Three pieces make up our PennyLane program:

- A **device** runs the circuit. `default.qubit` is a simulator; `wires=1` gives it one qubit.
- A **gate** changes the qubit's state. `Hadamard` prepares an equal superposition from `|0>`.
- A **QNode** connects a Python circuit function to the device. The `@qml.qnode(device)` line does this.

Qubits start in `|0>` each time the circuit runs. Wire labels start at 0 in these examples. `qml.probs` returns measurement probabilities in the order `[P(0), P(1)]`.


In [2]:
device = qml.device("default.qubit", wires=1)

@qml.qnode(device)
def first_circuit():
    qml.Hadamard(wires=0)
    return qml.probs(wires=0)

print(first_circuit())  # Expected: [0.5 0.5]


[0.5 0.5]


There is a 50% chance of measuring 0 and a 50% chance of measuring 1. This cell returns the exact probabilities calculated by the simulator, rather than a single random measurement result. Re-running it gives the same probabilities.

You can also print the circuit. `H` means Hadamard.


In [3]:
print(qml.draw(first_circuit)())


0: ──H─┤  Probs


## 3. Control the circuit with an angle

`RY` rotates the qubit's state about the y axis. The angle is in **radians**. Replace the fixed Hadamard gate with this adjustable gate.

For a qubit starting in `|0>`, the probability of measuring 1 after `RY(angle)` is `sin(angle / 2)**2`.


In [6]:
@qml.qnode(device)
def rotation_circuit(angle):
    qml.RY(angle, wires=0)
    return qml.probs(wires=0)

for angle in [0, math.pi / 2, math.pi]:
    print(f"angle = {angle:.2f}: {rotation_circuit(angle).round(3)}")


angle = 0.00: [1. 0.]
angle = 1.57: [0.5 0.5]
angle = 3.14: [0. 1.]


Expected probabilities:

| Angle | P(0) | P(1) |
| --- | --- | --- |
| 0 | 1 | 0 |
| pi / 2 | 0.5 | 0.5 |
| pi | 0 | 1 |

Tiny values close to zero may appear because of floating-point arithmetic.


## 4. Two qubits: create a Bell state

Use two wires. First apply Hadamard to qubit 0, then a **controlled-NOT** (`CNOT`). In `wires=[0, 1]`, qubit 0 is the control and qubit 1 is the target: the target flips when the control is 1.

Together these gates prepare the entangled state `(|00> + |11>) / sqrt(2)`.


In [5]:
two_qubit_device = qml.device("default.qubit", wires=2)

@qml.qnode(two_qubit_device)
def bell_circuit():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

print(qml.draw(bell_circuit)())
print(bell_circuit())  # Expected: [0.5 0.  0.  0.5]


0: ──H─╭●─┤ ╭Probs
1: ────╰X─┤ ╰Probs
[0.5 0.  0.  0.5]


The output order is `[P(00), P(01), P(10), P(11)]`. Measuring both qubits gives `00` or `11`, each with probability 50%. Their outcomes always match in this measurement basis.

## 5. Try it yourself

1. Remove the Hadamard gate from `first_circuit`. Predict the probabilities, then rerun its definition and call it.
2. Replace that gate with `qml.PauliX(wires=0)`. This flips `|0>` to `|1>`. What changes?
3. Call `rotation_circuit(math.pi / 3)`. Can you predict the probability of 1?
4. Remove the CNOT from `bell_circuit`. Which two outcomes are now possible?

Answers: 1. `[1, 0]`; 2. `[0, 1]`; 3. P(1) = 0.25; 4. `00` and `10`, each with probability 0.5.

## Learn more

- [Official PennyLane guide to circuits, devices, and QNodes](https://docs.pennylane.ai/en/stable/introduction/circuits.html)
- [Official qubit rotation tutorial](https://pennylane.ai/demos/tutorial_qubit_rotation): a next step for learning how to optimize circuit parameters.
